# GraphRAG Unit-Test Pipeline
A. Indexing phase: build the knowledge graph
1. Configure the PDF: Specify the PDF file path and a unique dataset ID (atomic-habits-pdf_unit_test_v1).
2. Extract PDF text: Read the PDF page by page and preserve page numbers and source metadata.
3. Create chunks: Split extracted text into smaller, overlapping chunks so each piece can be processed independently while retaining some context.
4. Extract entities and relationships: Use LLMGraphTransformer to convert each chunk into graph documents containing entities (nodes) and relationships (edges).
5. Prepare graph documents: Add dataset-specific IDs, preserve original entity names and readable display names, and update relationship endpoints.
6. Validate graph data: Check that relationships point to valid entities.
7. Save to Neo4j: Persist the prepared nodes and relationships, along with source information.
8. Inspect the stored graph: Retrieve nodes and relationships from Neo4j to verify that indexing worked.

B. Global indexing: understand the whole document

9. Load the graph into NetworkX: Bring the PDF's stored graph into memory for graph analysis.
10. Detect communities: Apply the Leiden algorithm to group closely connected entities into communities.
11. Inspect communities: Identify each community's entities, internal relationships and external relationships.
12. Prepare community report inputs: Organize community information into structured data for the LLM.
13. Generate community reports: Ask the LLM to summarize each community.
14. Save and connect reports: Store reports in Neo4j and connect them to their corresponding entities.

C. Query phase: answer questions

15. Load community reports: Retrieve the saved reports for global searching.
16. Perform Global Search: Split reports into batches, extract relevant findings from each batch (Map), and combine those findings into a final answer (Reduce).
17. Perform Local Search: Extract entities from the question, match them with Neo4j nodes, traverse their graph connections, convert retrieved paths into facts, and generate an answer.
18. Route the question: Use ScaleDown to classify the query as LOCAL or GLOBAL, then call the corresponding retrieval function.
19. Test the complete pipeline: Run sample questions and inspect intermediate results, retrieved evidence and final answers.


Complete Flow:
`PDF → Text → Chunks → Entities & Relationships → Neo4j Graph → Communities & Reports → Local / Global Search → Answer`


## Import and configure

In [2]:
import os, re, json, hashlib
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph

print("Imports loaded.")
print("Document = text + metadata; splitter = chunking; ChatOpenAI = model client;")
print("LLMGraphTransformer = graph extraction; Neo4jGraph = graph database connection.")


/home/ekkel-ai/momina/projects/graphrag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_73181/1094245414.py:7: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


Imports loaded.
Document = text + metadata; splitter = chunking; ChatOpenAI = model client;
LLMGraphTransformer = graph extraction; Neo4jGraph = graph database connection.


In [3]:
load_dotenv(override=True)
api_key = os.getenv("LLM_API_KEY")
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

missing = [
    name for name, value in {
        "LLM_API_KEY": api_key,
        "NEO4J_URI": neo4j_uri,
        "NEO4J_USERNAME": neo4j_username,
        "NEO4J_PASSWORD": neo4j_password,
    }.items() if not value
]
if missing:
    raise ValueError("Set these environment variables before continuing: " + ", ".join(missing))

model_name = os.getenv("LLM_MODEL", "gpt-4o-mini")
llm_kwargs = {"model": model_name, "api_key": api_key, "temperature": 0}
if os.getenv("LLM_BASE_URL"):
    llm_kwargs["base_url"] = os.getenv("LLM_BASE_URL")

llm = ChatOpenAI(**llm_kwargs)
graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
)

print(f"LLM client configured: {model_name}")
print(f"Neo4j connection configured for: {neo4j_uri}")
print("Credentials are not displayed.")


LLM client configured: openai/gpt-oss-120b
Neo4j connection configured for: neo4j+s://cb96a564.databases.neo4j.io
Credentials are not displayed.


In [4]:
import os
from langchain_anthropic import ChatAnthropic
load_dotenv(override=True)
api_key = os.getenv("ANTHROPIC_API_KEY")
model_name = os.getenv("ANTHROPIC_MODEL")
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY is missing")

if not model_name:
    raise ValueError("ANTHROPIC_MODEL is missing")

llm = ChatAnthropic(
    model=model_name,
    api_key=api_key,
    temperature=0,
    max_tokens=4096
)

print(f"LLM configured: {model_name}")

LLM configured: claude-haiku-4-5-20251001


In [5]:
PDF_DATASET_ID = "atomic-habits-pdf_unit_test_v1"

## Testing with a pdf: Atomic habits

In [6]:
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document

PDF_PATH = Path("/home/ekkel-ai/Downloads/atomic-habbits-chp1.pdf")
PDF_DATASET_ID = "atomic-habits-pdf_unit_test_v1"

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"
assert PDF_PATH.suffix.lower() == ".pdf", "Expected a PDF file"

## `Indexing phase:`

## pdf page extraction and Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)


reader = PdfReader(str(PDF_PATH))
pdf_documents = []

for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()

    if text:
        pdf_documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": PDF_PATH.name,
                    "dataset_id": PDF_DATASET_ID,
                    "page": page_number,
                },
            )
        )

pdf_chunks = text_splitter.split_documents(pdf_documents)

for index, chunk in enumerate(pdf_chunks):
    chunk.metadata["chunk_index"] = index

print("Pages with text:", len(pdf_documents))
print("Total chunks:", len(pdf_chunks))
print("First chunk metadata:", pdf_chunks[0].metadata if pdf_chunks else None)

Pages with text: 12
Total chunks: 42
First chunk metadata: {'source': 'atomic-habbits-chp1.pdf', 'dataset_id': 'atomic-habits-pdf_unit_test_v1', 'page': 1, 'chunk_index': 0}


## Extract Graph

In [14]:
transformer = LLMGraphTransformer(llm=llm)

pdf_graph_docs = transformer.convert_to_graph_documents(pdf_chunks)

print("Graph documents:", len(pdf_graph_docs))
print("Extracted nodes:", sum(len(gd.nodes) for gd in pdf_graph_docs))
print(
    "Extracted relationships:",
    sum(len(gd.relationships) for gd in pdf_graph_docs)
)

for index, gd in enumerate(pdf_graph_docs[:3]):
    print(f"\nChunk {index}")

    print("Nodes:")
    for node in gd.nodes:
        print(node.id, node.type, node.properties)

    print("Relationships:")
    for rel in gd.relationships:
        print(rel.source.id, "--", rel.type, "-->", rel.target.id)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

Graph documents: 42
Extracted nodes: 436
Extracted relationships: 277

Chunk 0
Nodes:
British Cycling Organization {}
Dave Brailsford Person {}
Great Britain Location {}
Olympic Games Event {}
Tour De France Event {}
1908 Date {}
2003 Date {}
Relationships:
Dave Brailsford -- EMPLOYED_BY --> British Cycling
British Cycling -- OPERATES_IN --> Great Britain

Chunk 1
Nodes:
Top Bike Manufacturer Europe Organization {}
Team Organization {}
Brailsford Person {}
British Cycling Organization {}
Aggregation Of Marginal Gains Concept {}
Relationships:
Top Bike Manufacturer Europe -- REFUSED_SALE --> Team
Brailsford -- EMPLOYED_BY --> British Cycling
Brailsford -- USES --> Aggregation Of Marginal Gains

Chunk 2
Nodes:
Brailsford Person {}
Coaches Person {}
Professional Cycling Team Organization {}
Bike Seats Object {}
Tires Object {}
Electrically Heated Overshorts Object {}
Biofeedback Sensors Object {}
Athlete Person {}
Fabrics Object {}
Wind Tunnel Object {}
Outdoor Riders Person {}
Indoor Rac

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Add Graph Docs to Neo4j with metadata

In [15]:
def prepare_pdf_graph_documents(graph_documents, dataset_id):
    prepared = []

    for gd in graph_documents:
        # Preserve source document metadata
        source_metadata = dict(gd.source.metadata or {})
        source_metadata["dataset_id"] = dataset_id

        source_document = Document(
            page_content=gd.source.page_content,
            metadata=source_metadata,
        )

        # Map original IDs to namespaced IDs
        id_mapping = {}

        for node in gd.nodes:
            original_id = str(node.id)
            namespaced_id = f"{dataset_id}::{original_id}"

            id_mapping[original_id] = namespaced_id

            node.properties = dict(node.properties or {})
            node.properties["display_name"] = (
                node.properties.get("name") or original_id
            )
            node.properties["original_entity_id"] = original_id
            node.properties["dataset_id"] = dataset_id

            node.id = namespaced_id

        # Update relationship endpoints as well
        for rel in gd.relationships:
            rel.properties = dict(rel.properties or {})
            rel.properties["dataset_id"] = dataset_id

            original_source_id = str(rel.source.id)
            original_target_id = str(rel.target.id)

            if original_source_id not in id_mapping:
                raise ValueError(
                    f"Source node not found: {original_source_id}"
                )

            if original_target_id not in id_mapping:
                raise ValueError(
                    f"Target node not found: {original_target_id}"
                )

            rel.source.id = id_mapping[original_source_id]
            rel.target.id = id_mapping[original_target_id]

        prepared.append(
            type(gd)(
                nodes=gd.nodes,
                relationships=gd.relationships,
                source=source_document,
            )
        )

    return prepared


In [16]:
# Prepare graph documents
prepared_pdf_graph_docs = prepare_pdf_graph_documents(
    pdf_graph_docs,
    PDF_DATASET_ID,
)

# Validate relationship endpoints
for index, gd in enumerate(prepared_pdf_graph_docs):
    node_ids = {str(node.id) for node in gd.nodes}

    for rel in gd.relationships:
        assert str(rel.source.id) in node_ids, (
            f"Missing source in chunk {index}: {rel.source.id}"
        )

        assert str(rel.target.id) in node_ids, (
            f"Missing target in chunk {index}: {rel.target.id}"
        )

print("Graph documents:", len(prepared_pdf_graph_docs))
print("Total nodes:", sum(len(gd.nodes) for gd in prepared_pdf_graph_docs))
print("Total relationships:", sum(len(gd.relationships) for gd in prepared_pdf_graph_docs))
print("All relationship endpoints validated!")

Graph documents: 42
Total nodes: 436
Total relationships: 277
All relationship endpoints validated!


In [17]:
graph.add_graph_documents(
    prepared_pdf_graph_docs,
    include_source=True,
)

print("PDF graph written to Neo4j.")

PDF graph written to Neo4j.


In [18]:
pdf_node_rows = graph.query(
    """
    MATCH (n)
    WHERE n.dataset_id = $dataset_id
    RETURN
        n.id AS id,
        n.display_name AS name,
        labels(n) AS labels,
        properties(n) AS properties
    ORDER BY name
    """,
    {"dataset_id": PDF_DATASET_ID},
)

pdf_edge_rows = graph.query(
    """
    MATCH (a)-[r]->(b)
    WHERE a.dataset_id = $dataset_id
      AND b.dataset_id = $dataset_id
      AND r.dataset_id = $dataset_id
    RETURN
        a.display_name AS source,
        type(r) AS relationship,
        b.display_name AS target,
        properties(r) AS properties
    ORDER BY source, relationship, target
    """,
    {"dataset_id": PDF_DATASET_ID},
)

print("PDF nodes:", len(pdf_node_rows))
print("PDF relationships:", len(pdf_edge_rows))

print("\nNodes:")
print(json.dumps(pdf_node_rows[:20], indent=2, ensure_ascii=False, default=str))

print("\nRelationships:")
print(json.dumps(pdf_edge_rows[:20], indent=2, ensure_ascii=False, default=str))

PDF nodes: 488
PDF relationships: 275

Nodes:
[
  {
    "id": "atomic-habits-pdf_unit_test_v1::1 Percent",
    "name": "1 Percent",
    "labels": [
      "Concept"
    ],
    "properties": {
      "dataset_id": "atomic-habits-pdf_unit_test_v1",
      "original_entity_id": "1 Percent",
      "id": "atomic-habits-pdf_unit_test_v1::1 Percent",
      "display_name": "1 Percent"
    }
  },
  {
    "id": "atomic-habits-pdf_unit_test_v1::1 Percent",
    "name": "1 Percent",
    "labels": [
      "Percentage"
    ],
    "properties": {
      "dataset_id": "atomic-habits-pdf_unit_test_v1",
      "original_entity_id": "1 Percent",
      "id": "atomic-habits-pdf_unit_test_v1::1 Percent",
      "display_name": "1 Percent"
    }
  },
  {
    "id": "atomic-habits-pdf_unit_test_v1::1 Percent Error",
    "name": "1 Percent Error",
    "labels": [
      "Concept"
    ],
    "properties": {
      "dataset_id": "atomic-habits-pdf_unit_test_v1",
      "original_entity_id": "1 Percent Error",
      "id": "

## `Community detection phase`
- install: `%pip install networkx igraph leidenalg`
- for the community building below the paragraph dataset is being shown to understand it easily. The same way did that for the pdf dataset.

## load graph into Networkx
- Neo4j dataset -> NetworkX graph

In [58]:
# Check community-detection dependencies
import importlib.util

required_packages = ["networkx", "igraph", "leidenalg"]

for package in required_packages:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'installed' if installed else 'MISSING'}")

networkx: installed
igraph: installed
leidenalg: installed


In [67]:
import networkx as nx

def load_dataset_graph(dataset_id):
    """Load one dataset from Neo4j into an undirected NetworkX graph."""

    if not dataset_id:
        raise ValueError("dataset_id must not be empty")

    G = nx.Graph()

    # 1. Load all nodes belonging to this dataset,
    node_rows = graph.query(
        """
        MATCH (n)
        WHERE n.dataset_id = $dataset_id
        RETURN labels(n) AS labels,
               n.id AS entity_id,
               properties(n) AS properties
        ORDER BY entity_id
        """,
        {"dataset_id": dataset_id},
    )

    for row in node_rows:
        entity_id = row["entity_id"]

        if entity_id is None:
            raise ValueError(
                f"Found a node without an 'id' in {dataset_id}"
            )

        G.add_node(
            entity_id,
            labels=row["labels"],
            properties=row["properties"],
            dataset_id=dataset_id,
        )

    # 2. Load relationships only between nodes in this dataset.
    edge_rows = graph.query(
        """
        MATCH (a)-[r]->(b)
        WHERE a.dataset_id = $dataset_id
          AND b.dataset_id = $dataset_id
          AND r.dataset_id = $dataset_id
        RETURN a.id AS source_id,
               b.id AS target_id,
               type(r) AS relationship_type,
               properties(r) AS properties
        ORDER BY source_id, target_id
        """,
        {"dataset_id": dataset_id},
    )

    for row in edge_rows:
        source_id = row["source_id"]
        target_id = row["target_id"]

        if source_id in G and target_id in G:
            G.add_edge(
                source_id,
                target_id,
                relationship_type=row["relationship_type"],
                properties=row["properties"],
            )

    print(f"Dataset: {dataset_id}")
    print(f"Nodes: {G.number_of_nodes()}")
    print(f"Edges: {G.number_of_edges()}")
    print(
        f"Connected components: "
        f"{nx.number_connected_components(G) if G.number_of_nodes() else 0}"
    )

    return G

In [ ]:
# COMMUNITY_DATASET_ID = PDF_DATASET_ID
COMMUNITY_DATASET_ID = PARAGRAPH_DATASET_ID
G = load_dataset_graph(COMMUNITY_DATASET_ID)

Dataset: garden_paragraph_unit_test_v1
Nodes: 20
Edges: 23
Connected components: 2


In [69]:
print("\nSample nodes:")
for node, data in list(G.nodes(data=True))[:5]:
    print(node, data["labels"])

print("\nSample relationships:")
for source, target, data in list(G.edges(data=True))[:5]:
    print(
        f"{source} --[{data['relationship_type']}]--> {target}"
    )


Sample nodes:
Anfaal ['Person']
Clean Tank ['Concept']
Community Garden ['Location']
Fresh Tomatoes ['Concept']
Garden ['Place']

Sample relationships:
Anfaal --[TEACHES]--> Momna
Clean Tank --[STORED_IN]--> Rainwater
Community Garden --[SUPPLIES]--> Fresh Tomatoes
Community Garden --[SUPPLIES]--> Spinach
Community Garden --[WATER]--> Momna


## detect communities
- NetworkX graph -> community assignments

In [102]:
import igraph as ig
import leidenalg
from collections import Counter

def detect_communities(G, resolution=0.3, seed=42):
    """Detect communities without modifying Neo4j or the input graph."""

    if G.number_of_nodes() == 0:
        raise ValueError("Graph has no nodes.")

    if G.number_of_edges() == 0:
        raise ValueError("Graph has no relationships to cluster.")

    # Keep a consistent mapping between NetworkX node IDs and igraph indices.
    nodes = list(G.nodes())
    node_to_index = {node: i for i, node in enumerate(nodes)}

    edges = [
        (node_to_index[source], node_to_index[target])
        for source, target in G.edges()
    ]

    ig_graph = ig.Graph(
        n=len(nodes),
        edges=edges,
        directed=False,
    )

    # Leiden groups densely connected nodes together.
    partition = leidenalg.find_partition(
        ig_graph,
        leidenalg.RBConfigurationVertexPartition,
        resolution_parameter=resolution,
        seed=seed,
    )

    # Record the resulting community ID on each NetworkX node.
    community_by_node = {}

    for community_id, members in enumerate(partition):
        for index in members:
            node_id = nodes[index]
            G.nodes[node_id]["community_id"] = community_id
            community_by_node[node_id] = community_id

    sizes = Counter(community_by_node.values())

    print(f"Nodes clustered: {len(nodes)}")
    print(f"Edges used: {len(edges)}")
    print(f"Communities found: {len(sizes)}")
    print(f"Community sizes: {dict(sorted(sizes.items()))}")

    return community_by_node

In [ ]:
community_by_node = detect_communities(G, resolution=0.5)

for community_id in sorted(set(community_by_node.values())):
    members = [
        node
        for node, cid in community_by_node.items()
        if cid == community_id
    ]

    print(f"\nCommunity {community_id} ({len(members)} entities)")
    for node in members:
        labels = G.nodes[node]["labels"]
        print(f"  - {node} [{', '.join(labels)}]")

Nodes clustered: 20
Edges used: 23
Communities found: 6
Community sizes: {0: 6, 1: 5, 2: 3, 3: 2, 4: 2, 5: 2}

Community 0 (6 entities)
  - Garden [Place]
  - Garden Vegetables [Vegetable]
  - Spinach [Food]
  - Tomatoes [Food]
  - Willow Village School [Organization]
  - Zainab [Person]

Community 1 (5 entities)
  - Anfaal [Person]
  - Lahore [Place]
  - Momna [Person]
  - Willow Community Garden [Place]
  - Willow Village [Place]

Community 2 (3 entities)
  - Clean Tank [Concept]
  - Rainwater [Concept]
  - Roof [Concept]

Community 3 (2 entities)
  - Community Garden [Location]
  - Fresh Tomatoes [Concept]

Community 4 (2 entities)
  - Garden Volunteers [Person]
  - School [Organization]

Community 5 (2 entities)
  - Lunch [Food]
  - School Children [Person]


## extract community details
- Graph + assignments -> entities and relationships per community

In [72]:
from collections import defaultdict

def inspect_communities(G, community_by_node):
    """Collect entities and relationships for each detected community."""

    communities = defaultdict(lambda: {
        "entities": [],
        "internal_relationships": [],
        "external_relationships": [],
    })

    # 1. Group entities by community.
    for node, community_id in community_by_node.items():
        communities[community_id]["entities"].append({
            "id": node,
            "labels": G.nodes[node]["labels"],
        })

    # 2. Examine every relationship.
    for source, target, data in G.edges(data=True):
        source_community = community_by_node[source]
        target_community = community_by_node[target]

        relationship = {
            "source": source,
            "relationship": data["relationship_type"],
            "target": target,
        }

        if source_community == target_community:
            communities[source_community][
                "internal_relationships"
            ].append(relationship)
        else:
            communities[source_community][
                "external_relationships"
            ].append(relationship)

    return dict(communities)

In [ ]:
#Graph + communities assignments -> entities and relationships per community
community_details = inspect_communities(G, community_by_node)

for community_id, details in community_details.items():
    print(f"\n{'=' * 50}")
    print(f"COMMUNITY {community_id}")

    print("\nEntities:")
    for entity in details["entities"]:
        print(f"  {entity['id']} ({', '.join(entity['labels'])})")

    print("\nInternal relationships:")
    for rel in details["internal_relationships"]:
        print(
            f"  {rel['source']} "
            f"--[{rel['relationship']}]--> "
            f"{rel['target']}"
        )

    print("\nExternal relationships:")
    for rel in details["external_relationships"]:
        print(
            f"  {rel['source']} "
            f"--[{rel['relationship']}]--> "
            f"{rel['target']}"
        )


COMMUNITY 0

Entities:
  Garden (Place)
  Garden Vegetables (Vegetable)
  Spinach (Food)
  Tomatoes (Food)
  Willow Village School (Organization)
  Zainab (Person)

Internal relationships:
  Garden --[SUPPLIES]--> Spinach
  Garden --[SUPPLIES]--> Tomatoes
  Garden --[SUPPLIES]--> Willow Village School
  Garden --[USES]--> Zainab
  Garden Vegetables --[USES]--> Zainab
  Spinach --[SUPPLIED_TO]--> Willow Village School
  Spinach --[USES]--> Zainab
  Tomatoes --[USES]--> Zainab
  Willow Village School --[COOK]--> Zainab

External relationships:

COMMUNITY 1

Entities:
  Anfaal (Person)
  Lahore (Place)
  Momna (Person)
  Willow Community Garden (Place)
  Willow Village (Place)

Internal relationships:
  Anfaal --[TEACHES]--> Momna
  Lahore --[NEAR]--> Willow Village
  Momna --[GROWS_IN]--> Willow Community Garden
  Momna --[LIVES_IN]--> Willow Village

External relationships:
  Momna --[COLLECTS]--> Rainwater

COMMUNITY 2

Entities:
  Clean Tank (Concept)
  Rainwater (Concept)
  Roof (Co

## Prepare the community report inputs
- Community details -> structured report data

In [79]:
def prepare_community_report_inputs(community_details):
    """Prepare structured facts for future LLM community reports."""

    report_inputs = {}

    for community_id, details in community_details.items():

        entities = details["entities"]
        relationships = details["internal_relationships"]

        report_inputs[community_id] = {
            "community_id": community_id,
            "entity_count": len(entities),
            "relationship_count": len(relationships),

            "entities": entities,
            "relationships": relationships,

            "external_relationships": details[
                "external_relationships"
            ],
        }

    return report_inputs

In [81]:
community_report_inputs = prepare_community_report_inputs(
    community_details=community_details
)

print(f"Prepared {len(community_report_inputs)} community inputs")

for item in community_report_inputs.values():
    print(f"\nCommunity: {item['community_id']}")
    print(f"Entities: {len(item['entities'])}")
    print(f"Relationships: {len(item['relationships'])}")
    print(f"External relationships: {len(item['external_relationships'])}")

Prepared 6 community inputs

Community: 0
Entities: 6
Relationships: 9
External relationships: 0

Community: 1
Entities: 5
Relationships: 4
External relationships: 1

Community: 2
Entities: 3
Relationships: 2
External relationships: 0

Community: 3
Entities: 2
Relationships: 1
External relationships: 3

Community: 4
Entities: 2
Relationships: 1
External relationships: 0

Community: 5
Entities: 2
Relationships: 1
External relationships: 1


## Generate the community report using llm
- Community data + LLM -> textual summaries

In [82]:
import json

def generate_community_report(community_input, llm):
    """Generate a factual summary for one graph community."""

    prompt = f"""
    You are analyzing a knowledge graph community.

    Use ONLY the graph facts provided below.
    Do not invent facts or relationships.
    If information is insufficient, explicitly say so.

    Community facts:
    {json.dumps(community_input, indent=2)}

    Generate a report with these sections:

    1. Community Overview
    2. Main Entities
    3. Important Relationships
    4. Key Insights

    Keep the report clear, concise, and factual.
    """

    response = llm.invoke(prompt)

    return response.content

In [83]:
community_reports = {}

for community_id, community_input in community_report_inputs.items():

    report = generate_community_report(
        community_input=community_input,
        llm=llm
    )

    community_reports[community_id] = report

    print(f"Generated report for community {community_id}")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generated report for community 0


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generated report for community 1


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generated report for community 2


... (3 outputs left out)

Generated report for community 4


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Generated report for community 5


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [84]:
first_community_id = next(iter(community_reports))
print(community_reports[first_community_id])

**Community Overview**  
- **Community ID:** 0  
- **Total Entities:** 6  
- **Total Relationships:** 9  
- The community links a place (Garden), a collection (Garden Vegetables), two food items (Spinach, Tomatoes), an organization (Willow Village School), and a person (Zainab).

---

**Main Entities**  

| Entity ID | Labels |
|-----------|--------|
| Garden | Place |
| Garden Vegetables | Vegetable |
| Spinach | Food |
| Tomatoes | Food |
| Willow Village School | Organization |
| Zainab | Person |

---

**Important Relationships**  

| Source | Relationship | Target |
|--------|--------------|--------|
| Garden | SUPPLIES | Spinach |
| Garden | SUPPLIES | Tomatoes |
| Garden | SUPPLIES | Willow Village School |
| Garden | USES | Zainab |
| Garden Vegetables | USES | Zainab |
| Spinach | SUPPLIED_TO | Willow Village School |
| Spinach | USES | Zainab |
| Tomatoes | USES | Zainab |
| Willow Village School | COOK | Zainab |

---

**Key Insights**  

1. **Central Role of the Garden** – 

## save community reports to neo4j
- Reports -> Neo4j CommunityReport nodes

In [104]:
def save_community_reports(community_reports, dataset_id, community_report_inputs):
    """Save generated community reports into Neo4j."""

    query = """
    UNWIND $reports AS report

    MERGE (c:CommunityReport {
        dataset_id: $dataset_id,
        community_id: report.community_id
    })

    SET c.summary = report.summary,
        c.entity_count = report.entity_count,
        c.relationship_count = report.relationship_count,
        c.updated_at = datetime()

    RETURN count(c) AS saved_count
    """

    reports_to_save = []

    for community_id, summary in community_reports.items():

        community_input = community_report_inputs[community_id]

        reports_to_save.append({
            "community_id": int(community_id),
            "summary": summary,
            "entity_count": community_input["entity_count"],
            "relationship_count": community_input["relationship_count"],
        })

    result = graph.query(
        query,
        {
            "dataset_id": dataset_id,
            "reports": reports_to_save,
        }
    )

    saved_count = result[0]["saved_count"]
    print(f"Saved {saved_count} community reports to Neo4j.")

    return saved_count

In [ ]:
saved_count = save_community_reports(
    community_reports=community_reports,
    dataset_id=COMMUNITY_DATASET_ID,
    community_report_inputs=community_report_inputs
)

Saved 6 community reports to Neo4j.


In [87]:
saved_reports = graph.query(
    """
    MATCH (c:CommunityReport)
    WHERE c.dataset_id = $dataset_id
    RETURN c.community_id AS community_id,
           c.entity_count AS entities,
           c.relationship_count AS relationships,
           c.summary AS summary
    ORDER BY c.community_id
    """,
    {"dataset_id": COMMUNITY_DATASET_ID}
)

print(f"Reports found in Neo4j: {len(saved_reports)}")

for report in saved_reports:
    print(f"\nCommunity: {report['community_id']}")
    print(f"Entities: {report['entities']}")
    print(f"Relationships: {report['relationships']}")
    print(f"Summary:\n{report['summary']}")

Reports found in Neo4j: 6

Community: 0
Entities: 6
Relationships: 9
Summary:
**Community Overview**  
- **Community ID:** 0  
- **Total Entities:** 6  
- **Total Relationships:** 9  
- The community links a place (Garden), a collection (Garden Vegetables), two food items (Spinach, Tomatoes), an organization (Willow Village School), and a person (Zainab).

---

**Main Entities**  

| Entity ID | Labels |
|-----------|--------|
| Garden | Place |
| Garden Vegetables | Vegetable |
| Spinach | Food |
| Tomatoes | Food |
| Willow Village School | Organization |
| Zainab | Person |

---

**Important Relationships**  

| Source | Relationship | Target |
|--------|--------------|--------|
| Garden | SUPPLIES | Spinach |
| Garden | SUPPLIES | Tomatoes |
| Garden | SUPPLIES | Willow Village School |
| Garden | USES | Zainab |
| Garden Vegetables | USES | Zainab |
| Spinach | SUPPLIED_TO | Willow Village School |
| Spinach | USES | Zainab |
| Tomatoes | USES | Zainab |
| Willow Village School | 

## Connect community reports to its entities
- Community membership -> CONTAINS_ENTITY edges

In [88]:
def connect_reports_to_entities(
    community_report_inputs,
    dataset_id
):
    """Connect each community report to its member graph entities."""

    query = """
    UNWIND $memberships AS membership

    MATCH (report:CommunityReport {
        dataset_id: $dataset_id,
        community_id: membership.community_id
    })

    MATCH (entity {id: membership.entity_id})
    WHERE entity.dataset_id = $dataset_id

    MERGE (report)-[:CONTAINS_ENTITY]->(entity)

    RETURN count(*) AS connected_count
    """

    memberships = []

    for community_id, details in community_report_inputs.items():
        for entity in details["entities"]:
            memberships.append({
                "community_id": int(community_id),
                "entity_id": entity["id"],
            })

    result = graph.query(
        query,
        {
            "dataset_id": dataset_id,
            "memberships": memberships,
        }
    )

    connected_count = result[0]["connected_count"]

    print(f"Connected {connected_count} community memberships.")

    return connected_count

In [89]:
connected_count = connect_reports_to_entities(
    community_report_inputs=community_report_inputs,
    dataset_id=COMMUNITY_DATASET_ID,
)

Connected 21 community memberships.


In [90]:
membership_check = graph.query(
    """
    MATCH (report:CommunityReport)-[:CONTAINS_ENTITY]->(entity)
    WHERE report.dataset_id = $dataset_id
    RETURN report.community_id AS community_id,
           labels(entity) AS entity_labels,
           entity.id AS entity_id
    ORDER BY community_id
    """,
    {"dataset_id": COMMUNITY_DATASET_ID}
)

print(f"Membership relationships: {len(membership_check)}")

for row in membership_check[:10]:
    print(row)

Membership relationships: 21
{'community_id': 0, 'entity_labels': ['Concept'], 'entity_id': 'Spinach'}
{'community_id': 0, 'entity_labels': ['Organization'], 'entity_id': 'Willow Village School'}
{'community_id': 0, 'entity_labels': ['Place'], 'entity_id': 'Garden'}
{'community_id': 0, 'entity_labels': ['Food'], 'entity_id': 'Tomatoes'}
{'community_id': 0, 'entity_labels': ['Food'], 'entity_id': 'Spinach'}
{'community_id': 0, 'entity_labels': ['Person'], 'entity_id': 'Zainab'}
{'community_id': 0, 'entity_labels': ['Vegetable'], 'entity_id': 'Garden Vegetables'}
{'community_id': 1, 'entity_labels': ['Person'], 'entity_id': 'Momna'}
{'community_id': 1, 'entity_labels': ['Place'], 'entity_id': 'Willow Village'}
{'community_id': 1, 'entity_labels': ['Place'], 'entity_id': 'Lahore'}


In [5]:
def filter_community_inputs(community_report_inputs):
    return {
        community_id: details
        for community_id, details in community_report_inputs.items()
        if details["entity_count"] >= 2
    }

## Build Communities Pipeline
- here build the communties for the pdf dataset combining all the above functions in single pipeline
- Dataset ID + LLM -> complete community-building process

In [113]:
def build_communities(dataset_id, llm):

    print(f"\nBuilding communities for: {dataset_id}")

    # 1. Load graph
    G = load_dataset_graph(dataset_id)

    # 2. Detect communities
    community_by_node = detect_communities(G)

    # 3. Inspect communities
    community_details = inspect_communities(
        G, community_by_node
    )

    # 4. Prepare report inputs
    community_report_inputs = prepare_community_report_inputs(
        community_details=community_details
    )
    community_report_inputs = filter_community_inputs(
        community_report_inputs
    )
    print("Communities eligible for reports:", len(community_report_inputs))
    # 5. Generate community reports
    community_reports = {}

    for community_id, community_input in community_report_inputs.items():
        community_reports[community_id] = generate_community_report(
            community_input=community_input,
            llm=llm
        )

    # 6. Save reports to Neo4j
    save_community_reports(
        community_reports,
        dataset_id,
        community_report_inputs
    )

    # 7. Connect reports to entities
    connected_count = connect_reports_to_entities(
        community_report_inputs,
        dataset_id
    )

    print("\nCommunity building completed!")
    print(f"Communities: {len(community_report_inputs)}")
    print(f"Reports: {len(community_reports)}")
    print(f"Entity connections: {connected_count}")

    return community_reports

In [119]:
atomic_habits_reports = build_communities(
    dataset_id=PDF_DATASET_ID,
    llm=llm
)


Building communities for: atomic-habits-pdf_unit_test_v1
Dataset: atomic-habits-pdf_unit_test_v1
Nodes: 450
Edges: 264
Connected components: 210
Nodes clustered: 450
Edges used: 264
Communities found: 213
Community sizes: {0: 44, 1: 38, 2: 34, 3: 33, 4: 8, 5: 8, 6: 7, 7: 6, 8: 5, 9: 5, 10: 5, 11: 4, 12: 4, 13: 4, 14: 4, 15: 4, 16: 3, 17: 3, 18: 3, 19: 3, 20: 3, 21: 2, 22: 2, 23: 2, 24: 2, 25: 2, 26: 2, 27: 2, 28: 2, 29: 2, 30: 2, 31: 2, 32: 2, 33: 2, 34: 2, 35: 2, 36: 2, 37: 2, 38: 2, 39: 2, 40: 2, 41: 2, 42: 2, 43: 2, 44: 2, 45: 2, 46: 2, 47: 2, 48: 2, 49: 2, 50: 2, 51: 1, 52: 1, 53: 1, 54: 1, 55: 1, 56: 1, 57: 1, 58: 1, 59: 1, 60: 1, 61: 1, 62: 1, 63: 1, 64: 1, 65: 1, 66: 1, 67: 1, 68: 1, 69: 1, 70: 1, 71: 1, 72: 1, 73: 1, 74: 1, 75: 1, 76: 1, 77: 1, 78: 1, 79: 1, 80: 1, 81: 1, 82: 1, 83: 1, 84: 1, 85: 1, 86: 1, 87: 1, 88: 1, 89: 1, 90: 1, 91: 1, 92: 1, 93: 1, 94: 1, 95: 1, 96: 1, 97: 1, 98: 1, 99: 1, 100: 1, 101: 1, 102: 1, 103: 1, 104: 1, 105: 1, 106: 1, 107: 1, 108: 1, 109: 1, 11

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

Saved 51 community reports to Neo4j.
Connected 324 community memberships.

Community building completed!
Communities: 51
Reports: 51
Entity connections: 324


## global Search using Community Reports
- Dataset ID -> saved community reports

In [13]:
def load_community_reports(dataset_id):
    """Load saved community reports from Neo4j."""

    query = """
    MATCH (c:CommunityReport {dataset_id: $dataset_id})
    RETURN c.community_id AS community_id,
           c.summary AS summary
    ORDER BY c.community_id
    """

    return graph.query(query, {"dataset_id": dataset_id})

In [14]:
community_reports = load_community_reports(PDF_DATASET_ID)

print("Reports loaded:", len(community_reports))
print(community_reports[0])

Reports loaded: 51
{'community_id': 0, 'summary': '**Atomic‑Habits‑PDF Community Report**  \n*Community ID: 0*  \n\n---\n\n### 1. Community Overview\n| Item | Value |\n|------|-------|\n| **Community ID** | 0 |\n| **Number of entities** | 44 |\n| **Number of internal relationships** | 48 |\n| **Number of external relationships** | 2 |\n| **Entity types represented** | Concept, Person, Object, Place, Language |\n\nThe graph captures a set of concepts and actors drawn from the *Atomic Habits* material and links them through a variety of semantic relations (e.g., *INFLUENCE*, *MAGNIFY*, *THINKS_AS*).\n\n---\n\n### 2. Main Entities  \n(Entities that appear most frequently as sources or targets in the relationship list.)\n\n| Entity | Labels | Notable connections (as source) |\n|--------|--------|---------------------------------|\n| **You** | Person | Target of many actions (e.g., *INFLUENCE* from Bad\u202fHabits, *BUILD* from Good\u202fHabits, *EAT* Unhealthy\u202fMeal, *STUDY* Mandarin, 

## Map: extract relevant findings from each report
- Question + report batch -> relevant findings

In [12]:

import json
from pydantic import BaseModel, Field


class CommunityFindings(BaseModel):
    findings: list[str] = Field(
        description="Relevant findings supported by the community reports"
    )


def map_community_report(question, reports, llm):
    """Extract relevant findings using structured LLM output."""

    prompt = f"""
    You are analyzing knowledge graph community reports.

    User question:
    {question}

    Community reports:
    {json.dumps(reports, indent=2, default=str)}

    Instructions:
    - Analyze all reports.
    - Extract only facts relevant to the question.
    - Preserve important entities and relationships.
    - Combine duplicate findings.
    - Do not invent information.
    - Ignore irrelevant information.
    - If no relevant information exists, return an empty findings list.
    - Include only findings supported by the reports.
    """

    structured_llm = llm.with_structured_output(CommunityFindings)

    result = structured_llm.invoke(prompt)

    return result.findings


## Reduce: combine the batch findings
- Findings -> final global answer

In [11]:
def reduce_community_findings(question, findings, llm):
    """Combine findings from all batches into a final answer."""

    if not findings:
        return "No relevant information was found in the community reports."

    prompt = f"""
    You are answering a question using knowledge graph findings.

    User question:
    {question}

    Findings from community reports:
    {json.dumps(findings, indent=2, default=str)}

    Instructions:
    - Combine related findings into a coherent answer.
    - Identify common themes and important relationships.
    - Remove duplicate information.
    - Preserve important details.
    - Do not introduce unsupported facts.
    - Clearly mention if the available evidence is insufficient.

    Provide a clear, structured answer.
    """

    response = llm.invoke(prompt)
    

    return response.content.strip()

In [ ]:
#Question + dataset -> global answer
def global_search(question, dataset_id, llm, batch_size=10):
    """Perform global search using batched community reports."""

    print("Starting global search...")

    # 1. Load community reports
    reports = load_community_reports(dataset_id)

    if not reports:
        return "No community reports found for this dataset."

    print(f"Loaded {len(reports)} community reports.")

    # 2. Divide reports into batches
    batches = [
        reports[i:i + batch_size]
        for i in range(0, len(reports), batch_size)
    ]

    print(f"Created {len(batches)} batches.")

    # 3. Map: process each batch
    all_findings = []

    for index, batch in enumerate(batches, start=1):

        print(f"Processing batch {index}/{len(batches)}...")

        findings = map_community_report(
            question=question,
            reports=batch,
            llm=llm
        )

        all_findings.extend(findings)

    print(f"Total findings extracted: {len(all_findings)}")

    # 4. Reduce: combine all findings
    answer = reduce_community_findings(
        question=question,
        findings=all_findings,
        llm=llm
    )

    print("Global search completed.")

    return answer

In [34]:
answer = global_search(
    question="What are the main findings discussed in this document?",
    dataset_id=PDF_DATASET_ID,
    llm=llm,
    batch_size=4
)

print(answer)

Starting global search...
Loaded 51 community reports.
Created 13 batches.
Processing batch 1/13...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Processing batch 2/13...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Processing batch 3/13...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


... (20 outputs left out)

Total findings extracted: 80


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Global search completed.
# Main Findings from the Atomic Habits Knowledge Graph

## Core Philosophy: Habits as Drivers of Personal Development

The document centers on how **habits function as the primary force shaping personal development and long-term outcomes**. Key insights include:

- **Atomic habits** are tiny, incremental changes that serve as building blocks for compound growth and remarkable results
- **Habits shape trajectory** through compounding effects—a consistent 1% daily improvement yields 37x better outcomes over one year, while 1% daily decline approaches zero
- **Progress is non-linear and fragile**, often resembling an ice cube that doesn't melt at typical temperatures, meaning visible change may not occur despite underlying progress

## Systems vs. Goals: A Critical Distinction

A central theme distinguishes between two approaches to success:

- **Systems are more reliable than isolated goals** for achieving long-term success
- **Goals create tension with happiness

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Query Classifier

In [9]:
import os
import requests

SCALEDOWN_URL = "https://api.scaledown.xyz/classify"


def classify_with_scaledown(question):
    api_key = os.getenv("SCALEDOWN_API_KEY")

    if not api_key:
        raise ValueError("SCALEDOWN_API_KEY environment variable is missing.")

    response = requests.post(
        SCALEDOWN_URL,
        headers={"x-api-key": api_key},
        json={
            "text": question,
            "labels": [
                {
                    "name": "LOCAL",
                    "rubric": """
                    Choose LOCAL when the question can be answered by
                    retrieving specific entities, facts, attributes,
                    relationships, or paths from the knowledge graph.

                    This includes:
                    - Finding a particular entity or its properties.
                    - Asking how two or more specific entities are connected.
                    - Following a relationship path between entities.
                    - Finding all graph records matching a specific condition.
                    - Looking up a specific fact, even if the question is
                      phrased as an explanation or contains several entities.

                    Examples:
                    - "Who is connected to Anfaal?"
                    - "How are Anfaal and Momna connected?"
                    - "What does the Community Garden supply?"
                    - "Which branches are located in Lahore?"
                    - "How many branches are in Lahore?"
                    - "Show all properties with three bedrooms."

                    Choose LOCAL when targeted graph retrieval is sufficient.
                    Do not choose GLOBAL merely because the question is long
                    or asks for multiple matching records.
                    """
                },
                {
                    "name": "GLOBAL",
                    "rubric": """
                    Choose GLOBAL when answering requires synthesizing
                    information across multiple community reports to
                    identify overall themes, recurring patterns, broad
                    insights, or comparisons across different parts of
                    the knowledge graph.

                    This includes:
                    - Identifying the main themes of an entire document
                      or knowledge domain.
                    - Summarizing recurring ideas across communities.
                    - Comparing strategies, topics, or patterns across
                      multiple community reports.
                    - Explaining how several broad aspects of a domain
                      relate to one another.
                    - Drawing an overall picture from distributed evidence
                      rather than retrieving a particular graph fact.

                    Examples:
                    - "What are the main principles of Atomic Habits?"
                    - "What recurring themes appear across the document?"
                    - "Compare strategies for building better habits."
                    - "What are the overall patterns in this community?"
                    - "What are the main trends across these property areas?"

                    Choose GLOBAL when cross-community synthesis is needed.
                    Do not choose GLOBAL for a specific entity lookup,
                    relationship lookup, or simple filtered record search.
                    """
                }
            ],
        },
        timeout=30,
    )

    response.raise_for_status()
    result = response.json()

    label = result.get("top_label")

    if not isinstance(label, str):
        raise ValueError(f"Missing top_label in ScaleDown response: {result}")

    label = label.strip().upper()

    if label not in {"LOCAL", "GLOBAL"}:
        raise ValueError(f"Unexpected ScaleDown label: {label}")

    return label


In [36]:
questions = [
    "Which entities are connected to identity-based habits?",
    "What are the main principles of Atomic Habits?",
    "Compare strategies for building better habits.",
]

for question in questions:
    print(question, "->", classify_with_scaledown(question))

Which entities are connected to identity-based habits? -> LOCAL
What are the main principles of Atomic Habits? -> GLOBAL
Compare strategies for building better habits. -> GLOBAL


In [8]:
def normalize_local_result(question, local_result):

    return {
        "question": question,
        "query_type": "LOCAL",
        "answer": local_result.get("answer", "No answer generated."),
        "sources": local_result.get("matched_entities", []),
        "metadata": {
            "dataset_id": local_result.get("dataset_id"),
            "query_entities": local_result.get("query_entities", []),
            "traversal_results": local_result.get("traversal_results", []),
            "graph_context": local_result.get("graph_context", [])
        }
    }

## Complete pipeline 

## `Query Retrieval Phase`
#### Used that same reterival phase logic from the paragraph str based graphrag testing, just Converted them in proper indenpdent functions to test.

In [7]:

from pydantic import BaseModel, Field
import json


#============================
# 1. Query entity extraction
#============================

class QueryEntities(BaseModel):
    entities: list[str] = Field(
        description=(
            "Individual searchable people, places, organizations, "
            "objects, and concepts mentioned or implied in the question."
        )
    )


entity_extractor = llm.with_structured_output(QueryEntities)


def extract_query_entities(question):
    query_result = entity_extractor.invoke([
        (
            "system",
            """
            Extract individual search entities from the question.

            Rules:
            1. Split possessive phrases into their underlying entities.
            2. Extract people, places, organizations, objects and concepts.
            3. Include important subjects and concepts.
            4. Return individual search terms, not long phrases.
            5. Do not answer the question or invent information.
            """
        ),
        ("human", question)
    ])

    query_entities = list(dict.fromkeys(
        e.strip()
        for e in query_result.entities
        if e.strip()
    ))

    return query_entities



def match_graph_entities(query_entities, dataset_id):
    matched_entities = []

    for entity in query_entities:
        results = graph.query(
            """
            MATCH (n)
            WHERE n.dataset_id = $dataset_id
              AND toLower(
                  coalesce(n.display_name, n.original_entity_id, n.id, '')
              ) CONTAINS toLower($entity)

            RETURN
                elementId(n) AS node_id,
                n.id AS name,
                n.display_name AS display_name,
                labels(n) AS labels

            LIMIT 10
            """,
            {
                "dataset_id": dataset_id,
                "entity": entity
            }
        )

        for result in results:
            matched_entities.append({
                "query_term": entity,
                "node_id": result["node_id"],
                "name": result["display_name"] or result["name"],
                "labels": result["labels"]
            })

    # Remove duplicate nodes matched by multiple terms
    unique = {}
    for item in matched_entities:
        unique[item["node_id"]] = item

    return list(unique.values())
#=================================================
# 3. Traverse the knowledge graph
#=================================================

def retrieve_graph_paths(matched_entities, dataset_id, max_hops=3):

    if not 1 <= max_hops <= 6:
        raise ValueError("max_hops must be between 1 and 3.")

    matched_node_ids = list(dict.fromkeys(
        item["node_id"] for item in matched_entities
    ))

    if not matched_node_ids:
        return []

    traversal_results = graph.query(
        f"""
        MATCH (seed)
        WHERE elementId(seed) IN $node_ids
          AND seed.dataset_id = $dataset_id

        MATCH path = (seed)-[*1..{max_hops}]-(neighbor)

        WHERE all(
            n IN nodes(path)
            WHERE n.dataset_id = $dataset_id
        )

        RETURN
            [
                n IN nodes(path) |
                {{
                    name: coalesce(n.display_name, n.id),
                    labels: labels(n)
                }}
            ] AS path_nodes,

            [
                r IN relationships(path) |
                {{
                    type: type(r),
                    properties: properties(r)
                }}
            ] AS path_relationships

        LIMIT 30
        """,
        {
            "node_ids": matched_node_ids,
            "dataset_id": dataset_id
        }
    )

    return traversal_results


#=================================================
# 4. Convert graph paths into readable facts
#=================================================

def format_graph_facts(traversal_results):

    graph_context = []

    for path in traversal_results:
        nodes = path["path_nodes"]
        relationships = path["path_relationships"]

        for i, rel in enumerate(relationships):
            source = nodes[i]["name"]
            target = nodes[i + 1]["name"]

            graph_context.append(
                f"{source} --[{rel['type']}]--> {target}"
            )

    # Remove duplicate facts
    graph_context = list(dict.fromkeys(graph_context))

    return graph_context


#=================================================
# 5. Generate an answer using retrieved graph facts
#=================================================

def generate_graph_answer(question, graph_context):

    if not graph_context:
        return (
            "I could not retrieve connected graph facts "
            "to answer this question."
        )

    context = "\n".join(graph_context)

    prompt = f"""
    Answer the question using the following facts retrieved
    from a knowledge graph.

    Retrieved facts:
    {context}

    Question:
    {question}

    Instructions:
    - Use only the retrieved facts as evidence.
    - Explain relevant relationships clearly.
    - Do not invent or assume missing information.
    - If the retrieved facts are insufficient, say so clearly.
    - Respond in a consise and factual manner. and try to keep the response under 7500 token limit

    Answer:
    """

    response = llm.invoke(prompt)

    return response.content


#=================================================
# 6. GraphRAG query function
#================================================

def run_graph_query(question, dataset_id, max_hops=3):

    # Step 1: Extract entities from question
    query_entities = extract_query_entities(question)

    # Step 2: Match entities with stored graph nodes
    matched_entities = match_graph_entities(
        query_entities,
        dataset_id
    )

    # Step 3: Traverse the graph
    traversal_results = retrieve_graph_paths(
        matched_entities,
        dataset_id,
        max_hops
    )

    # Step 4: Convert paths into facts
    graph_context = format_graph_facts(traversal_results)

    # Step 5: Generate grounded answer
    answer = generate_graph_answer(
        question,
        graph_context
    )

    # Return intermediate results as well as final answer
    return {
        "question": question,
        "dataset_id": dataset_id,
        "query_entities": query_entities,
        "matched_entities": matched_entities,
        "traversal_results": traversal_results,
        "graph_context": graph_context,
        "answer": answer
    }

print("OK")


OK


In [19]:
pdf_question = "what is aggregation of marginal gains?"
# pdf_question = "what are the surprising powers of atomic habits?"
pdf_result = run_graph_query(
    question=pdf_question,
    dataset_id=PDF_DATASET_ID,
    max_hops=3
)

print("ANSWER:")
print(pdf_result["answer"])

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


ANSWER:
# Aggregation of Marginal Gains

Based on the retrieved facts, **Aggregation of Marginal Gains** is a concept that is:

1. **Associated with Brailsford** - The knowledge graph shows that Aggregation of Marginal Gains uses Brailsford (likely referring to Dave Brailsford, a prominent cycling coach and team manager).

2. **Applied in Professional Cycling** - It is mentioned in contexts related to:
   - British Cycling
   - Professional Cycling Teams
   - Team performance optimization

3. **Involves Marginal Improvements** - The concept is connected to various small optimization areas including:
   - Equipment refinement (bike seats, tires, fabrics, racing suits)
   - Athlete performance (muscle temperature, biofeedback sensors, particular workouts)
   - Aerodynamic improvements (wind tunnel testing, making equipment "lighter and more aerodynamic")
   - Training methods (coaches, indoor and outdoor rider optimization)
   - The principle of achieving gains through "1 percent" improv

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [16]:
def route_query(question, dataset_id, llm):

    query_type = classify_with_scaledown(question)

    print(f"Query type: {query_type}")

    if query_type == "LOCAL":

        local_result = run_graph_query(question=question, dataset_id=dataset_id, max_hops=3) 

        return normalize_local_result(
            question=question,
            local_result=local_result
        )

    elif query_type == "GLOBAL":

        global_result = global_search(
            question=question,
            dataset_id=dataset_id,
            llm=llm
        )

        return {
            "question": question,
            "query_type": "GLOBAL",
            "answer": global_result,
            "sources": [],
            "metadata": {}
        }

    else:
        raise ValueError(f"Unknown query type: {query_type}")

In [17]:
question = "What are the main findings discussed in this document?"
result = route_query(
    question=question,
    dataset_id=PDF_DATASET_ID,
    llm=llm
)

print(result.keys())
print(result["query_type"])
print(result["answer"])

Query type: GLOBAL
Starting global search...
Loaded 51 community reports.
Created 6 batches.
Processing batch 1/6...


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Processing batch 2/6...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Processing batch 3/6...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


... (6 outputs left out)

Total findings extracted: 59


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Global search completed.
dict_keys(['question', 'query_type', 'answer', 'sources', 'metadata'])
GLOBAL
# Main Findings from the Document

## Core Themes

### 1. **The Power of Compound Growth and Small Changes**
- Small, incremental improvements (atomic habits) compound exponentially over time—a 1% daily improvement yields 37x better outcomes over a year
- Conversely, a 1% daily decline approaches zero, demonstrating the dual nature of compounding
- Small habits create big differences; big things derive from small beginnings
- Daily choices compound over years into significant life outcomes

### 2. **The Non-Linear Nature of Progress**
- Progress is obscured by the "Plateau of Latent Potential," where effort accumulates invisibly before producing visible results
- People experience a "Valley of Disappointment" where effort compounds without visible results (analogous to an ice cube not melting at room temperature until reaching 32 degrees)
- Critical thresholds act as transformation po

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Gradio interface

In [38]:

import gradio as gr
import json
import traceback


def test_graphrag(question, dataset_id):

    if not question or not question.strip():
        return (
            "Please enter a question.",
            "Not executed",
            [],
            {}
        )

    try:
        result = route_query(
            question=question.strip(),
            dataset_id=dataset_id,
            llm=llm
        )

        answer = result.get("answer", "No answer generated.")
        query_type = result.get("query_type", "UNKNOWN")
        sources = result.get("sources", [])
        metadata = result.get("metadata", {})

        # The answer area displays only the readable answer.
        return (
            str(answer),
            query_type,
            sources,
            metadata
        )

    except Exception as e:
        traceback.print_exc()

        return (
            f"**Error:** {str(e)}",
            "ERROR",
            [],
            {}
        )


with gr.Blocks(title="GraphRAG Query Tester") as demo:

    gr.Markdown(
        """
        # GraphRAG Query Tester
        """
    )

    with gr.Row():

        dataset = gr.Dropdown(
            choices=[
                "garden_paragraph_unit_test_v1",
                "atomic-habits-pdf_unit_test_v1"
            ],
            value="garden_paragraph_unit_test_v1",
            label="Select Dataset"
        )

    question = gr.Textbox(
        label="Your Question",
        placeholder="e.g. What are the main principles of Atomic Habits?",
        lines=3
    )

    submit = gr.Button(
        "Ask GraphRAG",
        variant="primary"
    )

    gr.Markdown("## Response")

    answer = gr.Markdown(
        label="Answer"
    )

    query_type = gr.Textbox(
        label="Selected Retrieval Strategy",
        interactive=False
    )

    with gr.Accordion("Sources", open=False):
        sources = gr.JSON(label="Retrieved Sources")

    with gr.Accordion("Technical Metadata", open=False):
        metadata = gr.JSON(label="Retrieval Metadata")

    submit.click(
        fn=test_graphrag,
        inputs=[question, dataset],
        outputs=[answer, query_type, sources, metadata]
    )

    question.submit(
        fn=test_graphrag,
        inputs=[question, dataset],
        outputs=[answer, query_type, sources, metadata]
    )

    gr.Examples(
        examples=[
            [
                "What are the main themes of the community?",
                "garden_paragraph_unit_test_v1"
            ],
            [
                "What are the main principles of Atomic Habits?",
                "atomic-habits-pdf_unit_test_v1"
            ],
            [
                "Which entities are connected to identity-based habits?",
                "atomic-habits-pdf_unit_test_v1"
            ]
        ],
        inputs=[question, dataset]
    )


demo.launch(
    server_name="127.0.0.1",
    server_port=8000,
    share=False,
    debug=True
)

* Running on local URL:  http://127.0.0.1:8000
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


## Cleanup query (Dataset Deletion)

In [5]:
# Delete only nodes belonging to this PDF dataset
# DETACH DELETE also removes their connected relationships.

graph.query(
    """
    MATCH (n {dataset_id: $dataset_id})
    DETACH DELETE n
    """,
    {"dataset_id": PDF_DATASET_ID},
)

print("Old PDF graph deleted.")

Old PDF graph deleted.


In [6]:
remaining = graph.query(
    """
    MATCH (n {dataset_id: $dataset_id})
    RETURN count(n) AS remaining_nodes
    """,
    {"dataset_id": PDF_DATASET_ID},
)

print("Remaining PDF nodes:", remaining[0]["remaining_nodes"])

Remaining PDF nodes: 0


In [ ]:
results = graph.query(
"""    MATCH (n {dataset_id: "garden_paragraph_unit_test_v1"})
OPTIONAL MATCH (n)-[r]-(m {dataset_id: "garden_paragraph_unit_test_v1"})
RETURN n, r, m"""
)

print("Records retrieved:", results)

Records retrieved: [{'n': {'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Momna'}, 'r': ({'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Momna'}, 'LIVES_IN', {'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Willow Village'}), 'm': {'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Willow Village'}}, {'n': {'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Momna'}, 'r': ({'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Anfaal'}, 'TEACHES', {'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Momna'}), 'm': {'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Anfaal'}}, {'n': {'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Momna'}, 'r': ({'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Momna'}, 'USES', {'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Rainwater'}), 'm': {'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Rainwater'}}, {'n': {'dataset_id': 'garden_paragraph_unit_test_v1', 'id': 'Momna'}, 'r': ({'dataset_id': 'gard

Query type: LOCAL


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

Query type: LOCAL


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

Query type: GLOBAL
Starting global search...
Loaded 51 community reports.
Created 6 batches.
Processing batch 1/6...


Traceback (most recent call last):
  File "/tmp/ipykernel_6434/381120269.py", line 17, in test_graphrag
    result = route_query(
             ^^^^^^^^^^^^
  File "/tmp/ipykernel_6434/1448715971.py", line 18, in route_query
    global_result = global_search(
                    ^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6434/3555907402.py", line 29, in global_search
    findings = map_community_report(
               ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6434/380916080.py", line 34, in map_community_report
    response = llm.invoke(prompt)
               ^^^^^^^^^^^^^^^^^^
  File "/home/ekkel-ai/momina/projects/graphrag/.venv/lib/python3.12/site-packages/langchain_core/language_models/chat_models.py", line 488, in invoke
    self.generate_prompt(
  File "/home/ekkel-ai/momina/projects/graphrag/.venv/lib/python3.12/site-packages/langchain_core/language_models/chat_models.py", line 1877, in generate_prompt
    return self.generate(prompt_messages, stop=stop, callbacks=callbacks, **kwar

Query type: GLOBAL
Starting global search...
Loaded 51 community reports.
Created 6 batches.
Processing batch 1/6...


Traceback (most recent call last):
  File "/tmp/ipykernel_6434/381120269.py", line 17, in test_graphrag
    result = route_query(
             ^^^^^^^^^^^^
  File "/tmp/ipykernel_6434/1448715971.py", line 18, in route_query
    global_result = global_search(
                    ^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6434/3555907402.py", line 29, in global_search
    findings = map_community_report(
               ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_6434/380916080.py", line 34, in map_community_report
    response = llm.invoke(prompt)
               ^^^^^^^^^^^^^^^^^^
  File "/home/ekkel-ai/momina/projects/graphrag/.venv/lib/python3.12/site-packages/langchain_core/language_models/chat_models.py", line 488, in invoke
    self.generate_prompt(
  File "/home/ekkel-ai/momina/projects/graphrag/.venv/lib/python3.12/site-packages/langchain_core/language_models/chat_models.py", line 1877, in generate_prompt
    return self.generate(prompt_messages, stop=stop, callbacks=callbacks, **kwar

: 